# Deploy — SageMaker Batch Transform

This is the stage the whole project is shaped around: **offline scoring of
the entire customer base**. A `Model.transformer` spins a serving container
on the inference-only image, streams the population through `model.joblib`,
and writes one predictions file per input file — then the container exits.
There is no endpoint to babysit; the "deployment" is a repeatable job.

The scored output is the retention campaign's worklist: every customer,
ranked by churn probability.


## Environment & local SageMaker session

In [ ]:
import os

# Kernel env on PATH so %%bash cells can call feast/aws/docker directly; the
# aws CLI and Feast binaries live in the workspace venv (/opt/mlops-venv).
os.environ["PATH"] = "/opt/mlops-venv/bin:" + os.environ["PATH"]
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# This project's own moto (see docker-compose.yml) — NOT the lab's shared one.
os.environ["MOTO_URL"] = "http://telco-customer-churn-moto:5000"

In [ ]:
import os
from pathlib import Path

import yaml
from sagemaker_local.config import LocalModeConfig
from sagemaker_local.patches import cleanup_stale_serving_containers
from sagemaker_local.session import make_local_session

# A killed run can leave a serving container bound to this project's host
# port; drop leftovers before we start (idempotent).
cleanup_stale_serving_containers()

# SAGEMAKER_LOCAL_REPO_PATH must resolve identically on the host and inside
# this container — local mode bind-mounts it into job/serving containers.
PROJECT_DIR = (
    Path(os.environ["SAGEMAKER_LOCAL_REPO_PATH"])
    / "projects"
    / "telco_customer_churn"
)

cfg = LocalModeConfig(
    **yaml.safe_load((PROJECT_DIR / "configs" / "local.yaml").read_text())
)
boto_session, sm_session = make_local_session(cfg)

TRAIN_IMAGE = "telco-customer-churn:train"
INFERENCE_IMAGE = "telco-customer-churn:inference"
print("s3 endpoint:", cfg.s3_endpoint_url, "| serving port:", cfg.serving_port)

## The infrastructure as code

One service is enough for a batch deployment: moto, hosting the model
artifacts and job inputs on S3.


In [ ]:
%%bash
cat ../docker-compose.yml

In [ ]:
%%bash
docker compose -f ../docker-compose.yml up -d

In [ ]:
%%bash
docker compose -f ../docker-compose.yml ps

## Produce the scoring input

The scoring `entity_df` is the full population — `scoring_entities.parquet`
carries every `customer_id` at the snapshot timestamp, no labels. The same
`get_historical_features` call that built the training set now builds the
transform input; the columns are written **headerless in contract order** —
the serving parser reads a positional matrix.


In [ ]:
import json

import numpy as np
import pandas as pd
from feast import FeatureStore

schema = json.loads(
    (PROJECT_DIR / "data" / "processed" / "feature_schema.json").read_text()
)
fs = FeatureStore(repo_path="../feature_repo")
entities = pd.read_parquet("../data/processed/scoring_entities.parquet")
scoring_df = fs.get_historical_features(
    entity_df=entities,
    features=[f"customer_subscription:{name}" for name in schema["features"]],
).to_df()

input_dir = PROJECT_DIR / "data" / "input"
input_dir.mkdir(parents=True, exist_ok=True)
input_csv = input_dir / "population.csv"
np.savetxt(
    input_csv,
    scoring_df[schema["features"]].astype("float64").to_numpy(),
    delimiter=",",
)
scoring_df.shape

## Record the input on S3

The transform job reads the local mount, but the campaign run is recorded:
the exact scored population lands next to the model artifacts on moto S3.


In [ ]:
%%bash
aws s3 cp ../data/input/population.csv     "s3://telco-customer-churn/batch/input/population.csv"     --endpoint-url "$MOTO_URL" && aws s3 ls "s3://telco-customer-churn/batch/input/" --endpoint-url "$MOTO_URL"

## Train then transform

The deploy stage re-fits the candidate quickly (the training stage explains
each step), then points the transformer at the produced `model.tar.gz`.


In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.model import Model

est = Estimator(
    entry_point="train.py",
    source_dir=str(PROJECT_DIR / "src" / "telco_customer_churn"),
    image_uri=TRAIN_IMAGE,
    role=cfg.role_arn,
    instance_type="local",
    instance_count=1,
    sagemaker_session=sm_session,
    output_path=f"s3://{cfg.bucket}/models",
    hyperparameters={"n_estimators": 100, "max_depth": 4},
)
est.fit({"train": f"s3://{cfg.bucket}/input/train/"})

In [ ]:
import shutil

output_dir = PROJECT_DIR / "data" / "output"
shutil.rmtree(output_dir, ignore_errors=True)

transformer = Model(
    image_uri=INFERENCE_IMAGE,
    model_data=est.model_data,
    role=cfg.role_arn,
    sagemaker_session=sm_session,
    name="telco-churn-batch-model",
).transformer(
    instance_count=1,
    instance_type="local",
    output_path=f"file:{output_dir}",
)
transformer.transform(
    f"file://{input_dir}",
    split_type="Line",
    content_type="text/csv",
)

## The scored population

One prediction per input row, in order — the `.out` file convention is the
same one real SageMaker Batch Transform uses for `Line` splits.


In [ ]:
predictions = json.loads((output_dir / "population.csv.out").read_text())
scored = scoring_df[["customer_id"]].assign(churn_probability=predictions)
scored = scored.sort_values("churn_probability", ascending=False)

top = scored.head(15)
print(f"scored {len(scored)} customers")
top

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(scored["churn_probability"], bins=40, color="#d65d0e")
plt.xlabel("churn probability")
plt.ylabel("customers")
plt.title("Scored population — churn risk distribution")
plt.show()

high_risk = scored[scored["churn_probability"] >= 0.5]
print(f"campaign list: {len(high_risk)} customers above threshold")

## Ship the result

The ranked list is the deploy stage's deliverable — persisted alongside the
input and model so the campaign run is fully auditable on S3.


In [ ]:
scored_path = PROJECT_DIR / "data" / "output" / "churn_scored.csv"
scored.to_csv(scored_path, index=False)
print(scored_path)

In [ ]:
%%bash
aws s3 cp ../data/output/churn_scored.csv     "s3://telco-customer-churn/batch/output/churn_scored.csv"     --endpoint-url "$MOTO_URL" && aws s3 ls "s3://telco-customer-churn/batch/output/" --endpoint-url "$MOTO_URL"

That is the whole deployment contract: **input population → Feast features
→ transform job → scored list on S3**. Re-running is the nightly schedule;
the next stage reviews whether the inputs are drifting.
